In [ ]:
import random

def attacker_success_probability_theoretical(q, z):
    """
    Computes the theoretical probability that an attacker with fraction q of stake
    can catch up from z blocks behind. For q < 0.5, we use the approximation:
        P ≈ (q/(1-q))^z
    For q >= 0.5, the attacker eventually wins with probability 1.
    """
    if q >= 0.5:
        return 1.0
    else:
        return (q / (1 - q)) ** z

def simulate_attack(q, z, num_simulations=100000):
    """
    Simulates many rounds of a "race" where at each block:
      - The attacker produces a block with probability q (reducing the deficit by 1)
      - The honest validators produce a block with probability 1-q (increasing the deficit by 1)
    The simulation counts how many times the attacker is able to overcome a deficit of z blocks.
    """
    success = 0
    for _ in range(num_simulations):
        deficit = z
        # Set a limit on the number of steps to avoid infinite loops in long-tail scenarios
        max_steps = 10000
        steps = 0
        while deficit > 0 and steps < max_steps:
            steps += 1
            if random.random() < q:
                deficit -= 1  # attacker wins a block, reducing the gap
            else:
                deficit += 1  # honest network wins a block, increasing the gap
        # If the attacker catches up (deficit <= 0), count it as a success
        if deficit <= 0:
            success += 1
    return success / num_simulations

# Example parameters:
q = 0.3   # Attacker's fraction of stake (30%)
z = 6     # Number of confirmations (i.e., the honest chain is 6 blocks ahead)

# We can easily simulate the above with different parameters.

theoretical_prob = attacker_success_probability_theoretical(q, z)
simulated_prob = simulate_attack(q, z)

print("For an attacker with {:.0%} of the stake and a confirmation depth of {}:".format(q, z))
print("  Theoretical attack success probability: {:.8f}".format(theoretical_prob))
print("  Simulated attack success probability:   {:.8f}".format(simulated_prob))

For an attacker with 30% of the stake and a confirmation depth of 6:
  Theoretical attack success probability: 0.00619640
  Simulated attack success probability:   0.00599000


In [2]:
import math
from scipy.stats import poisson

def pow_attack_probability(q, z):
    """
    Computes the probability of a successful attack (i.e., catching up from z blocks behind)
    in a Proof-of-Work system, following the model from the Bitcoin whitepaper.

    Parameters:
      q (float): The fraction of the total hash power controlled by the attacker (0 < q < 1).
      z (int): The number of confirmation blocks (i.e., the honest chain is z blocks ahead).

    Returns:
      float: The probability that an attacker with mining power q eventually catches up.
    
    Notes:
      - p is the fraction of honest miners, defined as p = 1 - q.
      - The parameter lambda is defined as: lambda = z * (q/p)
      - For k blocks mined by the attacker (while the honest network mines z blocks), the probability
        is given by the Poisson probability mass function (pmf).
      - If k >= z, the attacker is already ahead (and so the success probability for that term is 1).
      - For k < z, the probability of catching up is (q/p)^(z-k).
    """
    if q >= 0.5:
        # If the attacker has 50% or more hash power, they will eventually catch up with probability 1.
        return 1.0

    p = 1 - q
    lam = z * (q / p)
    probability = 0.0

    # Sum over k = 0 to z-1
    for k in range(z):
        pk = poisson.pmf(k, lam)
        probability += pk * ((q / p) ** (z - k))
    
    # Add the tail probability for k >= z
    probability += 1 - poisson.cdf(z - 1, lam)
    
    return probability

q = 0.3
# Confirmation depth (e.g., 6 confirmations)
z = 6

attack_prob = pow_attack_probability(q, z)
print("For an attacker with {:.0%} of the total hash power and {} confirmations:".format(q, z))
print("  The probability of catching up is approximately: {:.8f}".format(attack_prob))

For an attacker with 30% of the total hash power and 6 confirmations:
  The probability of catching up is approximately: 0.13211117


The above is much higher but that might be because it's way harder to control that much of the compute power? Ie if they have that much hash power it's that high but that's not realistic at all.